# 12 · Which features to keep

**The question.** DESIGN_QUESTIONS B8: which of the model's features earn their place in the chosen ensemble (XGBoost and a 100-tree random forest, rank-averaged; notebooks 10 and 11)? Fewer features mean less overfitting and simpler reasons (PSUPR Day 3), and WildGuard adds a second criterion of its own: a feature that shifts strongly between LEMIS and CITES may transfer badly.

**Fixed before looking:**
1. **Feature groups**, tested as wholes, not one feature at a time:
   - taxonomy: class, genus;
   - product: product group, unit group;
   - purpose and source: purpose, source, wild-source flag;
   - appendix;
   - countries: origin and exporter (country and region), re-export flag;
   - quantity: quantity, its deviation from peers, peer-group size;
   - route: the route's share of the genus's records and its volume;
   - rarity: rarity of the product × purpose × source combination, purpose and source shares in the genus;
   - consignment: number of distinct sources in the consignment (notebook 09).
2. **Embedded and filter importance** rank the groups (reported): XGBoost's total gain, and mutual information with the label on a sample of training line items.
3. **Leave-one-group-out:** the ensemble is retrained on 2000–2010 without each group in turn and scored on 2011. A group is **dropped** if removing it does not lower PR-AUC on listed line items significantly (the 95% paired-bootstrap interval of full − without includes zero). As a single pass, the droppable groups are then removed together and the reduced ensemble is checked against the full one; if it is significantly worse, only the least useful droppable group is removed.
4. **Shift (reported, tested later):** the population stability index (PSI) of each feature between LEMIS training line items and the CITES records rule D scores, computed with the LEMIS norms (S2.1). Strongly shifting groups that are kept are tested in the transfer check (C6): dropped there if the model transfers better without them.

This is a single pass rather than full iterative elimination, to keep the experiment to about half an hour.

In [1]:
import time

import duckdb
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import average_precision_score

import eda_setup as E
import experiment_setup as X_
import model_selection as M
from wildguard import config
from wildguard.features import CAT_SHARED, SHARED_FEATURES, FeatureBuilder

E.style()
t0 = time.time()
GROUPS = {"taxonomy": ["class", "genus_top"],
          "product": ["product_group", "unit_group"],
          "purpose and source": ["purpose", "source", "is_wild"],
          "appendix": ["appendix"],
          "countries": ["origin_region", "exporter_region", "exporter_top", "origin_top", "reexport_flag"],
          "quantity": ["log_qty", "qty_z", "peer_n_log"],
          "route": ["route_share", "route_n_log"],
          "rarity": ["combo_rarity", "purpose_share_in_genus", "source_share_in_genus"],
          "consignment": ["cons_sources"]}
FEATS = SHARED_FEATURES + ["cons_sources"]
assert sorted(sum(GROUPS.values(), [])) == sorted(FEATS)
df = X_.load_lemis()
cons = df.groupby("shipment_id")["source"].nunique().rename("cons_sources")
df = df[df["animal"]].join(cons, on="shipment_id").reset_index(drop=True)
tr, ev = df["year"].between(2000, 2010).to_numpy(), (df["year"] == 2011).to_numpy()
fb = FeatureBuilder(use_value=False).fit(df[tr])
F_tr, F_ev = fb.transform(df[tr]), fb.transform(df[ev])
y, y_ev = df["label"].to_numpy()[tr], df["label"].to_numpy()[ev]
listed, ships = df["sp_listed"].to_numpy()[ev], df["shipment_id"].to_numpy()[ev]
search = pd.read_parquet(config.PROCESSED_DIR / "experiments" / "model_search.parquet")
row = search[(search.candidate == "xgboost") & (search.config == 5)].iloc[0]
XGB_CFG = M.config_from_row(row.to_dict())
XGB_ROUNDS = int(row["best_rounds"])
print(f"features ready ({time.time() - t0:.0f}s); XGBoost {XGB_CFG}, {XGB_ROUNDS} rounds")


def ensemble_scores(feats, return_models=False):
    """The chosen ensemble trained on 2000-2010 with `feats`, scored on 2011."""
    base = M.Base().fit(F_tr, feats)
    Xtr, Xev = base.transform(F_tr), base.transform(F_ev)
    enc = M.Native(base).fit(Xtr)
    xgb = M.build_model("xgboost", XGB_CFG, enc, y, 37, XGB_ROUNDS).fit(enc.transform(Xtr), y)
    rf = RandomForestClassifier(n_estimators=100, max_depth=16, min_samples_leaf=5,
                                max_features="sqrt", n_jobs=8, random_state=37).fit(Xtr, y)
    s_x, s_r = xgb.predict_proba(enc.transform(Xev))[:, 1], rf.predict_proba(Xev)[:, 1]
    score = (pd.Series(s_x).rank(pct=True).to_numpy() + pd.Series(s_r).rank(pct=True).to_numpy()) / 2
    return (score, xgb, base) if return_models else score

features ready (19s); XGBoost {'imbalance': 'none', 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 20.0}, 357 rounds


## 1. Importance of each group

In [2]:
full, xgb_full, base_full = ensemble_scores(FEATS, return_models=True)
gain = xgb_full.get_booster().get_score(importance_type="total_gain")
names = [f"f{j}" for j in range(len(base_full.cats) + len(base_full.nums))]
col_of = dict(zip(names, base_full.cats + base_full.nums))
gain_by_feat = pd.Series({col_of[k]: v for k, v in gain.items() if k in col_of})
rng = np.random.default_rng(37)
samp = rng.choice(len(y), 50_000, replace=False)
Xs = base_full.transform(F_tr.iloc[samp])[:, :len(base_full.cats) + len(base_full.nums)]
disc = [f in CAT_SHARED for f in base_full.cats + base_full.nums]
mi = pd.Series(mutual_info_classif(Xs, y[samp], discrete_features=disc, random_state=37),
               index=base_full.cats + base_full.nums)
imp = pd.DataFrame({g: {"XGBoost gain share": gain_by_feat.reindex(f).fillna(0).sum() / gain_by_feat.sum(),
                        "mutual information": mi.reindex(f).sum()} for g, f in GROUPS.items()}).T
imp.sort_values("XGBoost gain share", ascending=False).style.format(
    {"XGBoost gain share": "{:.1%}", "mutual information": "{:.4f}"})

,XGBoost gain share,mutual information
purpose and source,29.0%,0.0363
countries,28.3%,0.0414
taxonomy,11.9%,0.0179
product,11.1%,0.0125
rarity,8.3%,0.0813
quantity,4.9%,0.0342
route,3.3%,0.0437
appendix,2.8%,0.0074
consignment,0.3%,0.0092


## 2. Leave one group out

In [3]:
scores = {"all groups": full}
for g, feats in GROUPS.items():
    scores[f"without {g}"] = ensemble_scores([f for f in FEATS if f not in feats])
    print(f"without {g}: done ({time.time() - t0:.0f}s)", flush=True)
loo = X_.pr_auc_bootstrap(y_ev[listed], {k: v[listed] for k, v in scores.items()}, ships[listed],
                          [("all groups", f"without {g}") for g in GROUPS])
loo["group"] = loo["b"].str.replace("without ", "")
loo = loo.set_index("group")[["pr_auc_b", "difference", "low_95", "high_95"]]
loo.columns = ["PR-AUC without the group", "loss (all − without)", "95% low", "95% high"]
loo["droppable"] = loo["95% low"] <= 0
loo.sort_values("loss (all − without)").style.format({c: "{:.4f}" for c in loo.columns[:4]})

without taxonomy: done (192s)


without product: done (278s)


without purpose and source: done (366s)


without appendix: done (453s)


without countries: done (536s)


without quantity: done (610s)


without route: done (694s)


without rarity: done (779s)


without consignment: done (868s)


,PR-AUC without the group,loss (all − without),95% low,95% high,droppable
group,,,,,
consignment,0.5715,0.0003,-0.0016,0.0021,True
product,0.5670,0.0048,0.0021,0.0074,False
taxonomy,0.5665,0.0053,0.0029,0.0077,False
rarity,0.5659,0.0060,0.0030,0.0090,False
appendix,0.5642,0.0075,0.0042,0.0109,False
route,0.5633,0.0085,0.0049,0.0121,False
purpose and source,0.5634,0.0085,0.0029,0.0142,False
quantity,0.5596,0.0123,0.0082,0.0168,False
countries,0.5488,0.0230,0.0148,0.0321,False


## 3. Dropping the droppable groups together

In [4]:
DROP = loo.index[loo["droppable"]].tolist()
if DROP:
    reduced = ensemble_scores([f for f in FEATS if not any(f in GROUPS[g] for g in DROP)])
    chk = X_.pr_auc_bootstrap(y_ev[listed], {"all groups": full[listed], "reduced": reduced[listed]},
                              ships[listed], [("all groups", "reduced")])
    display(chk.style.format({c: "{:.4f}" for c in ("pr_auc_a", "pr_auc_b", "difference", "low_95", "high_95")}))
    if chk["low_95"].iat[0] > 0:
        DROP = [loo.loc[DROP, "loss (all − without)"].idxmin()]
print("Groups dropped:", DROP or "none")
KEEP = [f for f in FEATS if not any(f in GROUPS[g] for g in DROP)]

,a,b,pr_auc_a,pr_auc_b,difference,low_95,high_95
0,all groups,reduced,0.5718,0.5715,0.0003,-0.0016,0.0021


Groups dropped: ['consignment']


## 4. How far each feature shifts between LEMIS and CITES

PSI between LEMIS training line items and a sample of the CITES records rule D scores (2015–2024, no source I or purpose L), with the LEMIS norms. Conventionally, under 0.1 is negligible and over 0.25 is large.

In [5]:
con = duckdb.connect()
cites = con.execute(f"""SELECT * FROM '{config.CITES_PARQUET}' WHERE year BETWEEN 2015 AND 2024
                        AND rule_d AND source <> 'I' AND purpose <> 'L' USING SAMPLE 500000 ROWS""").df()
ccons = con.execute(f"""SELECT coalesce(export_permit, 'record ' || shipment_id) AS consignment,
                               count(DISTINCT source) AS cons_sources
                        FROM '{config.CITES_PARQUET}' WHERE year BETWEEN 2015 AND 2024
                        AND reporter_type = 'E' GROUP BY 1""").df()
cites["consignment"] = cites["export_permit"].fillna("record " + cites["shipment_id"])
cites = cites.merge(ccons, on="consignment", how="left")
cites["cons_sources"] = cites["cons_sources"].fillna(1)
F_c = fb.transform(cites)


def psi(a, b, categorical):
    if categorical:
        p, q = a.astype(str).value_counts(normalize=True), b.astype(str).value_counts(normalize=True)
    else:
        edges = np.unique(np.nanquantile(a, np.linspace(0, 1, 11)[1:-1]))
        p = pd.Series(np.searchsorted(edges, a.fillna(-1e9), side="right")).value_counts(normalize=True)
        q = pd.Series(np.searchsorted(edges, b.fillna(-1e9), side="right")).value_counts(normalize=True)
    p, q = p.align(q, fill_value=0)
    return float(((p - q) * np.log((p + 1e-6) / (q + 1e-6))).sum())


shift = pd.Series({f: psi(F_tr[f], F_c[f], f in CAT_SHARED) for f in FEATS})
pd.DataFrame({"group": {f: g for g, fs in GROUPS.items() for f in fs}, "PSI": shift}).sort_values(
    "PSI", ascending=False).style.format({"PSI": "{:.2f}"})

,group,PSI
appendix,appendix,9.38
genus_top,taxonomy,5.05
class,taxonomy,2.04
exporter_top,countries,1.95
origin_top,countries,1.84
product_group,product,1.72
combo_rarity,rarity,1.48
purpose_share_in_genus,rarity,1.12
source_share_in_genus,rarity,0.98
exporter_region,countries,0.77


## Findings

| # | Finding |
|---|---|
| F1 | **Importance.** In XGBoost's gain, purpose and source (29%) and countries (28%) lead, then taxonomy (12%) and product (11%). Mutual information ranks the rarity features first (0.081): they are informative on their own but overlap with what the trees learn from purpose, source and taxon |
| F2 | **Leave one group out.** Removing any group except one lowers PR-AUC on listed 2011 line items significantly. The largest losses come from countries (−0.023) and quantity (−0.012); the smallest significant one from product (−0.005) |
| F3 | **The consignment feature adds nothing to the ensemble** (+0.0003, interval −0.0016 to +0.0021). Its narrow gain with XGBoost alone (notebook 09) disappears once the random forest is in. It is dropped |
| F4 | **Shift.** Almost every feature shifts strongly between LEMIS and the CITES records WildGuard scores (PSI above 0.25). The largest: appendix (9.4; CITES records are all listed species, most LEMIS line items are not), genus (5.1), class (2.0), countries (1.8–2.0), product group (1.7). Only the origin region, unit group, wild-source flag and consignment feature shift little |
| F5 | **Dropping a shifting group cannot pass the transfer rule** (DESIGN_QUESTIONS C6): a step is adopted only if it also helps on the listed held-out set, and F2 shows that removing any remaining group significantly hurts there. The shift is stated as a limitation, and measured as the domain shift of RQ5 |

**Decision (B8, 31 October 2026):** the ensemble uses the 21 shared features (`SHARED_FEATURES`); the consignment feature is not used.